# DeBERTa-v3 INT8 Quantization via OpenVINO SmoothQuant

**Author:** Konrad

**Approach:** mDeBERTa-v3 is the NLI half of the semantic grounder and is irreplaceable (no standard-attention NLI holds the signal), but it does not int8-quantize on ONNX Runtime - its disentangled attention produces activation outliers that naive int8 (dynamic 0.35, static 0.29) collapses. SmoothQuant migrates those outliers into the weights so activation-int8 survives. This notebook runs the working path end to end:

1. Export `mDeBERTa-v3-base-mnli-xnli` to an OpenVINO IR (fp32)
2. Quantize to int8 with NNCF SmoothQuant across a few `alpha` values (migration strength)
3. Parity-test each int8 model against the cached PyTorch max-over-chunks scores (target pearson >= 0.99)
4. Score the full gold with the best int8 model and re-fit the two-cross-encoder stack -> macro-F1 (the deployable gate, >= 0.782)

CPU-only (OpenVINO runtime); the reranker stays ONNX-Runtime int8 (already validated at 0.9975).

## Environment

CPU-only run - OpenVINO and onnxruntime, no GPU. Set before any heavy import.

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = ""        # CPU-only (OpenVINO + onnxruntime)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_OFFLINE"] = "1"             # models already cached

## Imports

All imports grouped: quantization toolchain (OpenVINO/NNCF via optimum-intel), the project scoring/stack helpers, and rich/tqdm for output.

In [ ]:
%load_ext autoreload
%autoreload 2

# Standard library
import gc
import time
from pathlib import Path

# Data processing
import numpy as np
import datasets
from scipy.stats import pearsonr, spearmanr

# OpenVINO / NNCF quantization (optimum-intel)
import openvino as ov
from optimum.intel import (OVModelForSequenceClassification, OVQuantizer,
                           OVConfig, OVQuantizationConfig)

# Transformers
from transformers import AutoTokenizer

# Stack meta-classifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

# Project
from grounding_models import load_gold, SCORES_DIR
from grounding_ensemble import best_macro

# Rich console output + progress
from rich import print as rprint
from tqdm.auto import tqdm

## Reproducibility

Fixed seed for the calibration and parity-sample selection.

In [ ]:
SEED = 42
np.random.seed(SEED)

## Configuration

All knobs for the quantization run. `alpha` is SmoothQuant's migration strength (higher pushes more of the activation outliers into the weights - the lever for DeBERTa's severe outliers). Two acceptance metrics: int8 parity vs fp32, and the end-to-end stack macro-F1.

In [ ]:
# Model
MODEL_ID = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"   # DeBERTa-v2 NLI, entailment = label index 0
ENT_IDX = 0                                            # entailment class in the 3-way head

# SmoothQuant sweep
ALPHAS = [0.5, 0.7, 0.9]                               # outlier-migration strength
N_CALIB = 150                                          # calibration (chunk, claim) pairs
CALIB_MAXLEN = 256                                     # calibration sequence cap

# Evaluation
N_SAMPLE = 24                                          # parity sample (half hallucination, half supported)
TARGET_PARITY = 0.99                                   # int8 vs PyTorch pearson
TARGET_MACRO_F1 = 0.782                                # one fold-std below the fp32 stack 0.796

# Paths
WORK = Path("/tmp/mde_quant")                          # OV IR outputs
FP32_DIR = WORK / "fp32"
WORK.mkdir(parents=True, exist_ok=True)

core = ov.Core()                                       # single OpenVINO runtime

rprint(f"""[bold cyan]Configuration[/bold cyan]
[dim]{"-" * 40}[/dim]
[bold]Model[/bold]
  Id: [cyan]{MODEL_ID}[/cyan]
  Entailment index: [yellow]{ENT_IDX}[/yellow]

[bold]SmoothQuant[/bold]
  Alphas: [yellow]{ALPHAS}[/yellow]
  Calibration pairs: [yellow]{N_CALIB}[/yellow] [dim](max_len {CALIB_MAXLEN})[/dim]

[bold]Acceptance[/bold]
  Parity (pearson vs fp32): [yellow]>= {TARGET_PARITY}[/yellow]
  Stack macro-F1: [yellow]>= {TARGET_MACRO_F1}[/yellow] [dim](fp32 stack 0.796)[/dim]

[bold]Paths[/bold]
  Work dir: [cyan]{WORK}[/cyan]

[bold]Runtime[/bold]
  Device: [green]CPU[/green] [dim](OpenVINO {ov.__version__})[/dim]
""")

## Data Loading

Loads the verified gold and the cached PyTorch scores (the parity reference and the reranker half of the stack), builds the SmoothQuant calibration set from real claim x chunk pairs, and draws a balanced parity sample.

In [ ]:
recs = load_gold()
y = np.array([r["label"] for r in recs])
cached_nli = np.load(SCORES_DIR / "MoritzLaurer__mDeBERTa-v3-base-mnli-xnli.npy")  # fp32 reference
cached_bge = np.load(SCORES_DIR / "BAAI__bge-reranker-v2-m3.npy")                  # reranker (stack)
tok = AutoTokenizer.from_pretrained(MODEL_ID)

# balanced parity sample
half = N_SAMPLE // 2
sample_idx = np.sort(np.concatenate([
    np.random.choice(np.where(y == 0)[0], half, replace=False),
    np.random.choice(np.where(y == 1)[0], half, replace=False)]))

# calibration: N_CALIB (chunk, claim) pairs, same pairing as the NLI scoring (premise=chunk)
cidx = np.random.choice(len(recs), N_CALIB, replace=False)
_enc = tok([recs[i]["chunks"][0] for i in cidx], [recs[i]["claim"] for i in cidx],
           truncation=True, max_length=CALIB_MAXLEN, padding="max_length")
calib = datasets.Dataset.from_dict({"input_ids": _enc["input_ids"],
                                    "attention_mask": _enc["attention_mask"]})

rprint(f"""[white]Data[/white]
  Gold records: [yellow]{len(recs)}[/yellow] [dim]({int((y==0).sum())} hallucination / {int((y==1).sum())} supported)[/dim]
  Parity sample: [yellow]{len(sample_idx)}[/yellow]
  Calibration pairs: [yellow]{len(calib)}[/yellow]
""")

## Helpers

`ov_nli_scores` mirrors the PyTorch NLI scoring exactly - pairs (chunk, claim), softmax entailment, max over chunks - on an OpenVINO compiled model. `parity` reports agreement with the fp32 reference.

In [ ]:
def _compile(xml_path):
    # THROUGHPUT hint -> OpenVINO picks multiple CPU streams for the async queue
    return core.compile_model(core.read_model(str(xml_path)), "CPU",
                              {"PERFORMANCE_HINT": "THROUGHPUT"})


def ov_nli_scores(xml_path, records, bs=16, progress=False):
    """Per-record max entailment prob over chunks, on an OpenVINO int8/fp32 IR.

    Throughput path: flatten every (chunk, claim) pair, pipeline them through an
    AsyncInferQueue across CPU streams, reduce to the per-record max entailment.
    """
    cm = _compile(xml_path)
    names = [i.get_any_name() for i in cm.inputs]
    pairs, owner = [], []
    for k, r in enumerate(records):
        for c in r["chunks"]:
            pairs.append((c, r["claim"])); owner.append(k)
    best = np.full(len(records), -1e9)

    def cb(req, ud):
        (idxs,) = ud
        logits = req.get_output_tensor(0).data
        ex = np.exp(logits - logits.max(1, keepdims=True))
        s = (ex / ex.sum(1, keepdims=True))[:, ENT_IDX]
        for m, oi in enumerate(idxs):
            if s[m] > best[oi]:
                best[oi] = s[m]

    q = ov.AsyncInferQueue(cm, cm.get_property("OPTIMAL_NUMBER_OF_INFER_REQUESTS"))
    q.set_callback(cb)
    rng = range(0, len(pairs), bs)
    for j in (tqdm(rng, desc="OV NLI scoring") if progress else rng):
        sub, ow = pairs[j:j + bs], owner[j:j + bs]
        e = tok([p[0] for p in sub], [p[1] for p in sub], padding=True,
                truncation=True, max_length=512, return_tensors="np")
        feed = {"input_ids": e["input_ids"].astype(np.int64),
                "attention_mask": e["attention_mask"].astype(np.int64)}
        q.start_async({n: feed[n] for n in names if n in feed}, (ow,))
    q.wait_all()
    return best


def parity(scores, ref):
    return dict(pearson=float(pearsonr(scores, ref)[0]),
                spearman=float(spearmanr(scores, ref)[0]),
                max_abs=float(np.abs(scores - ref).max()),
                std=float(scores.std()))

## Export fp32 OpenVINO IR

One-time export of the PyTorch checkpoint to an OpenVINO IR; SmoothQuant quantizes from this. Skipped if already present.

In [ ]:
if not (FP32_DIR / "openvino_model.xml").exists():
    t = time.time()
    base = OVModelForSequenceClassification.from_pretrained(MODEL_ID, export=True)
    base.save_pretrained(FP32_DIR)
    del base; gc.collect()
    rprint(f"[green]exported fp32 IR[/green] in [yellow]{time.time()-t:.0f}s[/yellow]")
fp32_mb = (FP32_DIR / "openvino_model.bin").stat().st_size / 1e6
rprint(f"fp32 IR size: [yellow]{fp32_mb:.0f} MB[/yellow]")

## Quantize with SmoothQuant (alpha sweep)

For each `alpha`, run NNCF int8 PTQ with SmoothQuant and parity-test against the fp32 reference. NNCF recurses into DeBERTa's 12 `If` subgraphs (the disentangled-attention control flow) - this is the step that the ONNX-Runtime SmoothQuant could not handle. Memory is freed between alphas.

In [ ]:
def quantize_smoothquant(alpha, out_dir):
    model = OVModelForSequenceClassification.from_pretrained(FP32_DIR)
    quantizer = OVQuantizer.from_pretrained(model)
    cfg = OVConfig(quantization_config=OVQuantizationConfig(
        bits=8, smooth_quant_alpha=alpha, num_samples=N_CALIB))
    quantizer.quantize(calibration_dataset=calib, save_directory=out_dir, ov_config=cfg)
    del model, quantizer; gc.collect()


results = []
for alpha in ALPHAS:
    sd = WORK / f"sq_a{int(alpha * 100)}"
    t = time.time()
    if not (sd / "openvino_model.xml").exists():
        quantize_smoothquant(alpha, sd)
    scores = ov_nli_scores(sd / "openvino_model.xml", [recs[i] for i in sample_idx])
    p = parity(scores, cached_nli[sample_idx])
    size_mb = (sd / "openvino_model.bin").stat().st_size / 1e6
    results.append(dict(alpha=alpha, size_mb=size_mb, secs=time.time() - t, dir=str(sd), **p))
    rprint(f"  alpha=[yellow]{alpha}[/yellow]  size=[cyan]{size_mb:.0f} MB[/cyan]  "
           f"pearson=[{'green' if p['pearson']>=TARGET_PARITY else 'yellow'}]{p['pearson']:.4f}[/]  "
           f"std={p['std']:.3f}  [dim]({time.time()-t:.0f}s)[/dim]")
    gc.collect()

## Parity Results

The int8 parity against fp32 for each alpha. Baselines (ONNX Runtime, same model): dynamic 0.35, static 0.29, mixed-precision 0.754, weight-only-4bit 0.90. SmoothQuant should beat all of them.

In [ ]:
best = max(results, key=lambda r: r["pearson"])
rows = "\n".join(
    f"  alpha [yellow]{r['alpha']}[/yellow]: pearson "
    f"[{'green' if r['pearson']>=TARGET_PARITY else 'yellow'}]{r['pearson']:.4f}[/]  "
    f"max|d| {r['max_abs']:.3f}  size [cyan]{r['size_mb']:.0f} MB[/cyan]"
    + ("  [green](best)[/green]" if r is best else "")
    for r in results)
rprint(f"""[bold cyan]SmoothQuant int8 parity vs fp32[/bold cyan]
[dim]{"-"*52}[/dim]
{rows}
[dim]baselines: ORT dynamic 0.35 / static 0.29 / mixed 0.754 / wo4bit 0.90[/dim]

Best: alpha [yellow]{best['alpha']}[/yellow] at [green]{best['pearson']:.4f}[/green], [cyan]{best['size_mb']:.0f} MB[/cyan]
""")

## Deployable gate - stack macro-F1

Parity is a proxy; the real gate is whether the two-cross-encoder stack {reranker, NLI} holds macro-F1 with the int8 NLI. Score the full gold with the best int8 model, then re-fit the logistic out-of-fold and compare to the fp32 stack (0.796). >= 0.782 means the int8 NLI is deployable.

In [ ]:
import json as _json

INT8_CACHE = SCORES_DIR / "mDeBERTa-v3-int8-sq.npy"
INT8_META = SCORES_DIR / "mDeBERTa-v3-int8-sq.json"

# full-gold int8 scoring is ~90 min on CPU; always reuse the cache when present (built by
# scripts/score_int8_fullgold.py), and align the reported best alpha to the cache's
if INT8_CACHE.exists():
    nli_int8 = np.load(INT8_CACHE)
    if INT8_META.exists():
        ca = _json.loads(INT8_META.read_text())["alpha"]
        best = next((r for r in results if abs(r["alpha"] - ca) < 1e-9), best)
else:
    nli_int8 = ov_nli_scores(Path(best["dir"]) / "openvino_model.xml", recs, progress=True)
    np.save(INT8_CACHE, nli_int8)

skf = StratifiedKFold(5, shuffle=True, random_state=42)
def stack_macro(nli_scores):
    X = np.column_stack([cached_bge, nli_scores])
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0))
    p = cross_val_predict(clf, X, y, cv=skf, method="predict_proba")[:, 1]
    return best_macro(p, y)[0], roc_auc_score(y, p)

mf1_int8, auc_int8 = stack_macro(nli_int8)
mf1_fp32, auc_fp32 = stack_macro(cached_nli)
ok = mf1_int8 >= TARGET_MACRO_F1

rprint(f"""[bold cyan]Stack macro-F1 (out-of-fold, full gold)[/bold cyan]
[dim]{"-"*48}[/dim]
  fp32 NLI : macro-F1 [yellow]{mf1_fp32:.3f}[/yellow]  AUC {auc_fp32:.3f}
  int8 NLI : macro-F1 [{'green' if ok else 'red'}]{mf1_int8:.3f}[/]  AUC {auc_int8:.3f}  [dim](alpha {best['alpha']})[/dim]
  Gate (>= {TARGET_MACRO_F1}): [{'green' if ok else 'red'}]{'PASS' if ok else 'FAIL'}[/]
""")

## Summary

Verdict for the CPU/Lambda NLI: the best int8 SmoothQuant model, its size, parity, and whether the stack holds.

In [ ]:
verdict = ("DEPLOYABLE - OpenVINO int8 SmoothQuant NLI holds the stack"
           if mf1_int8 >= TARGET_MACRO_F1 else
           "below gate - fall back to mDeBERTa fp32 on CPU")
rprint(f"""[bold]DeBERTa-v3 int8 on CPU - result[/bold]
[dim]{"-"*52}[/dim]
  Method: [cyan]OpenVINO / NNCF SmoothQuant[/cyan] (alpha [yellow]{best['alpha']}[/yellow])
  Size: [green]{best['size_mb']:.0f} MB[/green] [dim](fp32 {fp32_mb:.0f} MB, ~{fp32_mb/best['size_mb']:.1f}x smaller)[/dim]
  int8 parity vs fp32: [yellow]{best['pearson']:.4f}[/yellow]
  Stack macro-F1: [{'green' if mf1_int8>=TARGET_MACRO_F1 else 'red'}]{mf1_int8:.3f}[/] [dim](fp32 {mf1_fp32:.3f}, gate {TARGET_MACRO_F1})[/dim]
  Reranker: ONNX-Runtime int8 (unchanged, parity 0.9975)

  [bold]{verdict}[/bold]
""")